# Static rows and a fixed-window neural language model

Read Chapter 10. The paper trace and three-dimensional static map are authored teaching assumptions, not trained Word2Vec. The four-dimensional embeddings below are actually learned by a tiny NNLM. Retrieval success and next-token likelihood have separate evaluation sets and denominators.

In [1]:
from pathlib import Path
import sys, json, math
candidates = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in candidates if (p / "data/part-i/ngram.json").is_file()
             and (p / "src/config/book.mjs").is_file()), None)
if root is None:
    raise FileNotFoundError("book repository boundary not found")
sys.path.insert(0, str(root / "code/part-ii"))
print("Repository fixtures found")

Repository fixtures found


In [2]:
import torch
from nnlm import paper_trace, run, events, WindowLM, tensors
trace = paper_trace()
print("Authored paper trace:", trace)
assert trace["embedded_shape"] == [1, 2, 2]
assert math.isclose(trace["loss"], math.log(2), abs_tol=1e-12)

Authored paper trace: {'ids_shape': [1, 2], 'embedded_shape': [1, 2, 2], 'combined': [[1.0, 0.0, 0.0, 1.0]], 'hidden': [[1.0, 1.0]], 'logits': [[0.0, 0.0, 1.0986122886681098, 0.0]], 'probabilities': [[0.16666666666666666, 0.16666666666666666, 0.5, 0.16666666666666666]], 'loss': 0.6931471805599453}


## CPU training and independently counted targets

A fresh seeded model trains on 20 events, uses eight validation events for selection, and evaluates the selected parameters on eight test events. The trigram reference probabilities are supplied as independent integer fractions. Test values never select an update.

In [3]:
measured = run(write=False)
reference = json.loads((root / "data/part-ii/nnlm-run.json").read_text())
fixture = json.loads((root / "data/part-ii/nnlm-data.json").read_text())
assert measured["selected_step"] == 10
for method in ["count_lm", "nnlm"]:
    actual = measured["test_once_after_selection"][method]
    expected = reference["test_once_after_selection"][method]
    assert actual["events"] == 8
    assert math.isclose(actual["mean_loss"], expected["mean_loss"], abs_tol=1e-9)
    print(method, "mean loss:", actual["mean_loss"], "perplexity:", actual["perplexity"], "correct:", actual["correct"])
fractions = [n/d for n, d in fixture["test_count_target_fractions"]]
assert measured["test_once_after_selection"]["count_lm"]["target_probabilities"] == fractions
assert measured["test_once_after_selection"]["nnlm"]["correct"] == 5
assert measured["test_once_after_selection"]["count_lm"]["correct"] == 6

{
  "selected_step": 10,
  "train_loss": 1.2580410905050692,
  "validation_loss": 1.4542136735502054,
  "test": {
    "count_lm": {
      "target_probabilities": [
        0.2857142857142857,
        0.25,
        0.18181818181818182,
        0.1,
        0.2,
        0.2857142857142857,
        0.08333333333333333,
        0.2
      ],
      "predictions": [
        "beijing",
        "lodging",
        "lodging",
        "750",
        "EOS",
        "beijing",
        "lodging",
        "EOS"
      ],
      "mean_loss": 1.7003669947499123,
      "perplexity": 5.475956670356062,
      "events": 8,
      "correct": 6
    },
    "nnlm": {
      "mean_loss": 1.4244286318523989,
      "perplexity": 4.15548285378354,
      "correct": 5,
      "events": 8,
      "target_losses": [
        0.46708173914918505,
        1.579025834149149,
        1.6149389559860798,
        2.986624662516385,
        0.3067485205407024,
        0.46708173914918505,
        3.1021681368382183,
        0.871759

## Retrieval on the unchanged KA-0 queries

Pooling input embeddings is a declared reuse of a language-model representation, not an independently trained retrieval model. Record all ranking changes and failures, including the authored map's date/approval collisions and the NNLM's missing synonyms.

In [4]:
for method, locales in measured["retrieval"].items():
    for locale, result in locales.items():
        print(method, locale, "dimension:", result["dimension"], "success:", result["success_count"], "/", result["query_count"])
        for row in result["results"]:
            print(row)
assert measured["retrieval"]["authored_static"]["en"]["success_count"] == 4
assert measured["retrieval"]["nnlm_input_mean"]["en"]["success_count"] == 7

authored_static en dimension: 3 success: 4 / 9
{'scores': [1.0, 1.0, 0.9486832980505138, 0.0, 0.0], 'ranking': ['travel-v1', 'travel-v2', 'rail-faq-v1', 'status-faq-v1', 'approval-faq-v1'], 'prediction': 'travel-v1', 'query_id': 'current-lodging', 'success': False}
{'scores': [0.7071067811865475, 0.7071067811865476, 0.6708203932499369, 0.7071067811865475, 0.7071067811865475], 'ranking': ['travel-v1', 'travel-v2', 'status-faq-v1', 'approval-faq-v1', 'rail-faq-v1'], 'prediction': 'travel-v1', 'query_id': 'geographic-status', 'success': False}
{'scores': [1.0, 1.0, 0.9486832980505138, 0.0, 0.0], 'ranking': ['travel-v1', 'travel-v2', 'rail-faq-v1', 'status-faq-v1', 'approval-faq-v1'], 'prediction': 'travel-v1', 'query_id': 'explicit-current', 'success': False}
{'scores': [1.0, 1.0, 0.9486832980505138, 0.0, 0.0], 'ranking': ['travel-v1', 'travel-v2', 'rail-faq-v1', 'status-faq-v1', 'approval-faq-v1'], 'prediction': 'travel-v1', 'query_id': 'synonym', 'success': True}
{'scores': [1.0, 1.0, 0

## Transfer: identical visible windows

Use a separate five-symbol diagnostic. The red/blue clues differ earlier in the sequence, but both final windows contain the same IDs. Any deterministic network receiving only those IDs returns the same distribution. This is an information boundary, not a claim about an unrun red/blue training experiment.

In [5]:
red_prefix = [0] + [2] * 6 + [3, 4]
blue_prefix = [1] + [2] * 6 + [3, 4]
assert red_prefix != blue_prefix
assert red_prefix[-2:] == blue_prefix[-2:] == [3, 4]
torch.manual_seed(31)
diagnostic = WindowLM(5, 5).double().eval()
with torch.no_grad():
    output = diagnostic(torch.tensor([red_prefix[-2:], blue_prefix[-2:]], dtype=torch.long))
assert torch.equal(output[0], output[1])
print("identical visible windows:", red_prefix[-2:], blue_prefix[-2:])
print("identical logits:", output.tolist())

identical visible windows: [3, 4] [3, 4]
identical logits: [[0.42830515201452046, -0.17955748153715356, 0.38126673909232833, 0.2410021109430306, -0.06874728147004665], [0.42830515201452046, -0.17955748153715356, 0.38126673909232833, 0.2410021109430306, -0.06874728147004665]]
